# 🎙️ Chatterbox Turbo — Airi Wake Word Dataset Generator
### Voice Cloning TTS · Google Colab

Generates synthetic `.wav` samples of **"Hello Airi"** using **Chatterbox Turbo** by Resemble AI — a 350M-parameter model with paralinguistic tags, zero-shot voice cloning, and a 1-step decoder for fast inference.

---

| Feature | Detail |
|---|---|
| **Model** | ChatterboxTurboTTS (ResembleAI) |
| **License** | MIT |
| **Voice source** | Zero-shot cloning from reference audio |
| **Paralinguistic tags** | `[laugh]` `[chuckle]` `[cough]` built-in |
| **Model size** | ~350M parameters |
| **Native sample rate** | 24 000 Hz |
| **Output format** | 22 050 Hz mono WAV (pipeline standard) |
| **GPU** | Required — T4 recommended |

> ⚠️ Chatterbox embeds an imperceptible **PerTh watermark** in all output. This does **not** affect wake word training.

---

## 📋 Quick Start — Run in order

| Step | Action |
|---|---|
| 1 | ▶️ **Cell 1** — install all software |
| 2 | 🔄 `Runtime → Restart session` — **mandatory** after install |
| 3 | ▶️ **Cell 2** — set configuration |
| 4 | ▶️ **Cell 3** — upload or download reference voices |
| 5 | ▶️ **Cell 4** — load the model |
| 6 | ▶️ **Cell 5** — generate dataset |
| 7 | ▶️ **Cell 6** — preview a sample |
| 8 | ▶️ **Cell 7** — download your files |

> 💡 Set your runtime **before** starting: `Runtime → Change runtime type → T4 GPU`

---


---
## ⚙️ Step 1 of 2 — Install PyTorch

Installs a pinned, Colab-compatible version of PyTorch before Chatterbox is added.
This must run **first** so that `numpy` and CUDA bindings are in place.

> ⏱️ Takes ~1–2 minutes. Output is suppressed (`-q`). Wait for the cell to finish before proceeding.

---


In [ ]:
*INSTALLING TORCH*

In [ ]:
!pip install -q torch==2.6.0 torchaudio==2.6.0 torchvision==0.21.0

---
## ⚙️ Step 2 of 2 — Install Chatterbox & Dependencies

**Why this specific install order?**

Chatterbox depends on `pkuseg` (Chinese text segmentation), which requires `numpy` to be present at build time. The correct order is: `numpy` → `pkuseg` → `chatterbox-tts`. Running in any other order causes silent failures.

**After this cell finishes:** When you see `ALL INSTALLS COMPLETE`, go to `Runtime → Restart session`. Do **not** re-run this cell in the same session.

> ⏱️ Takes **3–5 minutes**. The long output is normal — just let it run.

---


In [ ]:
!pip install -q chatterbox-tts --upgrade

---
## 🔧 Cell 2 — Configuration

Edit the values below before running any other cells. Everything else in the notebook uses these settings.

| Variable | What it controls |
|---|---|
| `WAKE_WORD` | The phrase to synthesize |
| `TARGET_SAMPLES` | Number of `.wav` files to generate |
| `MIN_DURATION` / `MAX_DURATION` | Clip length filter (seconds) |
| `EXAGGERATION` | Emotion intensity: `0.0` = flat · `0.5` = natural · `1.0` = very expressive |
| `CFG_WEIGHT` | Voice clone tightness: `0.3` = loose · `0.7` = tight clone |

**Leave everything else as-is.**

> 💡 **Duration tip:** Chatterbox Turbo output length varies. `MIN=0.8, MAX=4.0` gives the lowest rejection rate. Tighter windows like `1.0–1.3` will discard many samples.

---


In [ ]:
import os, torch
from pathlib import Path
import numpy as np

# ============================================================
#  YOU CAN CHANGE THESE VALUES
# ============================================================
WAKE_WORD       = 'Hello Airi'   # The phrase to synthesize
TARGET_SAMPLES  = 10            # Start with 100; increase to 600 when working
MIN_DURATION    = 0.8            # Minimum clip length in seconds
MAX_DURATION    = 2.0            # Maximum clip length in seconds
BATCH_SIZE      = 10             # Clips per progress update
RANDOM_SEED     = 42             # Change each run for variety
EXAGGERATION    = 0.5            # Emotion intensity: 0.0 (flat) to 1.0 (expressive)
CFG_WEIGHT      = 0.5            # Voice clone strength: 0.3 (loose) to 0.7 (tight)
# ============================================================

# Set automatically — do not change
OUTPUT_DIR   = '/content/chatterbox_output'
SAMPLES_DIR  = '/content/ref_voices'
NATIVE_SR    = 24000   # Chatterbox Turbo native output rate
TARGET_SR    = 22050   # Normalise all output to this (pipeline standard)

# Auto-detect GPU
if torch.cuda.is_available():
    DEVICE = 'cuda'
    print(f'GPU detected: {torch.cuda.get_device_name(0)}')
    print('Generation will be fast (up to 6x real-time on T4).')
else:
    DEVICE = 'cpu'
    print('No GPU — running on CPU (slower).')
    print('Tip: Runtime -> Change runtime type -> T4 GPU')

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
Path(SAMPLES_DIR).mkdir(parents=True, exist_ok=True)

print()
print('Configuration set!')
print(f'  Wake word      : {WAKE_WORD!r}')
print(f'  Target samples : {TARGET_SAMPLES}')
print(f'  Duration range : {MIN_DURATION}s to {MAX_DURATION}s')
print(f'  Output SR      : {TARGET_SR} Hz (normalised from {NATIVE_SR} Hz)')
print(f'  Exaggeration   : {EXAGGERATION}')
print(f'  CFG weight     : {CFG_WEIGHT}')
print(f'  Device         : {DEVICE.upper()}')
print(f'  Output folder  : {OUTPUT_DIR}')


GPU detected: Tesla T4
Generation will be fast (up to 6x real-time on T4).

Configuration set!
  Wake word      : 'Hello Airi'
  Target samples : 10
  Duration range : 0.8s to 2.0s
  Output SR      : 22050 Hz (normalised from 24000 Hz)
  Exaggeration   : 0.5
  CFG weight     : 0.5
  Device         : CUDA
  Output folder  : /content/chatterbox_output


---
## 🎤 Cell 3 — Get Reference Voices

Chatterbox Turbo is a **voice cloning** model — it needs short reference audio clips (5–15 seconds of clean speech) to clone a voice from.

| Option | When to use |
|---|---|
| **Option A** *(recommended)* | Downloads sample voices from the NeuTTS repo — ready to use out of the box |
| **Option B** | Upload your own `.wav` files (5–15 sec, clean speech, no background noise) |

> 💡 More reference voices = more diversity in your generated dataset.

---


In [ ]:
# ── OPTION A: Download sample voices ─────────────────────────────────────
import shutil, subprocess
from pathlib import Path

REPO_DIR = '/tmp/ref_voice_repo'
shutil.rmtree(REPO_DIR, ignore_errors=True)  # Clean any previous attempt

print('Downloading sample reference voices...')
result = subprocess.run(
    ['git', 'clone', '--depth=1', '--filter=blob:none', '--sparse',
     'https://github.com/neuphonic/neutts-air.git', REPO_DIR],
    capture_output=True, text=True
)

if result.returncode != 0:
    print('Clone failed:', result.stderr[-300:])
    print('Please use Option B below to upload your own voice files.')
else:
    subprocess.run(
        ['git', '-C', REPO_DIR, 'sparse-checkout', 'set', 'samples'],
        capture_output=True
    )
    copied = 0
    for f in (Path(REPO_DIR) / 'samples').glob('*.wav'):
        shutil.copy(f, Path(SAMPLES_DIR) / f.name)
        print(f'  Copied: {f.name}')
        copied += 1
    print(f'\nDone! {copied} voice file(s) ready.')

ref_wavs = list(Path(SAMPLES_DIR).glob('*.wav'))
print(f'\nReference voices available: {len(ref_wavs)}')
for w in ref_wavs:
    print(f'  - {w.name}')


  Copied: greta.wav
  Copied: jo.wav
  Copied: juliette.wav
  Copied: dave.wav
  Copied: mateo.wav

Done! 5 voice file(s) ready.

Reference voices available: 5
  - greta.wav
  - jo.wav
  - juliette.wav
  - dave.wav
  - mateo.wav


In [ ]:
# ── OPTION B: Upload your own voice files ────────────────────────────────
# Run this INSTEAD of Option A if you have your own voice clips.
# Each file: .wav format, 5-15 seconds, clear speech, no background noise.

from google.colab import files
from pathlib import Path

print('Upload one or more .wav files (5-15 seconds of clear speech each).')
uploaded = files.upload()

for fname, data in uploaded.items():
    dest = Path(SAMPLES_DIR) / fname
    dest.write_bytes(data)
    print(f'  Saved: {fname}')

ref_wavs = list(Path(SAMPLES_DIR).glob('*.wav'))
print(f'\nTotal reference voices: {len(ref_wavs)}')


---
## 📦 Cell 4 — Load Chatterbox Turbo Model

Downloads and loads the model weights from HuggingFace into GPU memory.

| Run | What happens |
|---|---|
| **First run** | Downloads ~2–3 GB of model files — takes a few minutes |
| **Same session** | Weights already cached — loads instantly |

Just run the cell and wait for the confirmation message.

---


In [ ]:
from pathlib import Path

# Verify reference voices exist (you can keep this)
ref_wavs = list(Path(SAMPLES_DIR).glob('*.wav'))
if not ref_wavs:
    raise FileNotFoundError(
        f'No .wav files found in {SAMPLES_DIR}!\n'
        'Please run Cell 3 first.'
    )

print(f'Reference voices confirmed: {len(ref_wavs)} file(s)')
for w in ref_wavs:
    print(f'  - {w.name}')
print()

# ✅ Correct import (NOT turbo)
from chatterbox.tts import ChatterboxTTS

print(f'Loading Chatterbox model on {DEVICE.upper()}...')
print()

model = ChatterboxTTS.from_pretrained(device=DEVICE)

print('✅ Model loaded!')
NATIVE_SR = 24000  # default for chatterbox

# (optional — keep for consistency)
REF_VOICES = [str(w) for w in ref_wavs]
print(f'Ready with {len(REF_VOICES)} reference file(s).')
print('Run next cell to start generating!')

Reference voices confirmed: 5 file(s)
  - greta.wav
  - jo.wav
  - juliette.wav
  - dave.wav
  - mateo.wav

Loading Chatterbox model on CUDA...



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


ve.safetensors:   0%|          | 0.00/5.70M [00:00<?, ?B/s]

t3_cfg.safetensors:   0%|          | 0.00/2.13G [00:00<?, ?B/s]

s3gen.safetensors:   0%|          | 0.00/1.06G [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

conds.pt:   0%|          | 0.00/107k [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/diffusers/models/lora.py:393: FutureWarning: `LoRACompatibleLinear` is deprecated and will be removed in version 1.0.0. Use of `LoRACompatibleLinear` is deprecated. Please switch to PEFT backend by installing PEFT: `pip install peft`.
  deprecate("LoRACompatibleLinear", "1.0.0", deprecation_message)


loaded PerthNet (Implicit) at step 250,000
✅ Model loaded!
Ready with 5 reference file(s).
Run next cell to start generating!


---
## 🚀 Cell 5 — Generate Dataset

For each sample, the pipeline:

1. Picks a random reference voice
2. Randomly varies `exaggeration` and `cfg_weight` per sample for diversity
3. Synthesizes the wake word via voice cloning
4. Applies pitch shift and speed change (post-synthesis augmentation)
5. Normalises to 22 050 Hz mono WAV
6. Checks clip length — rejects if outside the duration window
7. Saves accepted clips to `OUTPUT_DIR`

**Reading the progress output:**

| Status | Meaning |
|---|---|
| `OK` ✅ | Sample accepted and saved |
| `SKIP` | Wrong length — discarded (normal; widen `MAX_DURATION` to reduce) |
| `ERR` | Generation error — rare |

> ⏱️ Chatterbox Turbo is fast — expect **~3–8 seconds per sample** on a T4 GPU.

---


In [ ]:
import random, time, soundfile as sf
from pathlib import Path
import numpy as np

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# ── Variation pools ─────────────────────────────────────────────
SPEED_RANGE = [0.85, 0.9, 1.0, 1.05, 1.1]

EMOTION_STYLES = ['neutral', 'happy', 'sad', 'angry',
                  'calm', 'excited', 'fearful']

# ── Safe audio helpers (no scipy) ───────────────────────────────
def change_speed(audio: np.ndarray, speed: float) -> np.ndarray:
    if speed == 1.0:
        return audio
    idx = np.round(np.arange(0, len(audio), speed)).astype(int)
    return audio[idx[idx < len(audio)]]

def add_emotion(audio: np.ndarray, emotion: str, sr: int) -> np.ndarray:
    scale = {
        'happy': 1.1,
        'excited': 1.15,
        'angry': 1.2,
        'sad': 0.9,
        'calm': 0.9
    }.get(emotion, 1.0)

    audio = audio * scale

    if emotion == 'fearful':
        tr = 1 + 0.1 * np.sin(2 * np.pi * 5 * np.arange(len(audio)) / sr)
        audio = audio * tr

    mx = np.abs(audio).max()
    return audio / mx * 0.95 if mx > 0 else audio


# ── Generation loop ─────────────────────────────────────────────
output_path      = Path(OUTPUT_DIR)
valid_samples    = 0
rejected_samples = 0
total_generated  = 0
epoch            = 1
start_time       = time.time()

print('=' * 60)
print('  AIRI WAKE WORD GENERATION — CHATTERBOX')
print('=' * 60)
print(f'  Wake word       : {WAKE_WORD!r}')
print(f'  Target samples  : {TARGET_SAMPLES}')
print(f'  Duration range  : {MIN_DURATION}s to {MAX_DURATION}s')
print(f'  Device          : {DEVICE.upper()}')
print(f'  Sample Rate     : {NATIVE_SR} Hz')
print('=' * 60)

while valid_samples < TARGET_SAMPLES:
    remaining  = TARGET_SAMPLES - valid_samples
    batch_size = min(BATCH_SIZE, remaining)

    print(f'\n--- Epoch {epoch} | {valid_samples}/{TARGET_SAMPLES} done | {remaining} to go ---')

    bv = br = 0

    for i in range(batch_size):
        speed   = random.choice(SPEED_RANGE)
        emotion = random.choice(EMOTION_STYLES)

        try:
            # ── Generate audio ─────────────────────────────
            wav_tensor = model.generate(WAKE_WORD)
            audio = wav_tensor.squeeze().cpu().numpy().astype(np.float32)

            # ── Post-processing ────────────────────────────
            audio = change_speed(audio, speed)
            audio = add_emotion(audio, emotion, NATIVE_SR)

            # ── Duration gate ──────────────────────────────
            dur = len(audio) / NATIVE_SR

            if not (MIN_DURATION <= dur <= MAX_DURATION):
                br += 1
                print(f'  SKIP [{i+1:>2}/{batch_size}] {dur:.2f}s outside range')
            else:
                ts = int(time.time() * 1000)
                fname = f"airi_{ts}_{i}_{emotion}_spd{speed:.2f}.wav"

                sf.write(str(output_path / fname), audio, NATIVE_SR)

                valid_samples += 1
                bv += 1

                print(f"  OK  [{i+1:>2}/{batch_size}] "
                      f"emo={emotion:<8} spd={speed:.2f} dur={dur:.2f}s")

        except Exception as e:
            br += 1
            print(f'  ERR [{i+1:>2}/{batch_size}] {e}')

        total_generated += 1

    rejected_samples += br
    epoch += 1

    elapsed = time.time() - start_time
    rate    = valid_samples / elapsed if elapsed > 0 else 0
    eta     = (TARGET_SAMPLES - valid_samples) / rate if rate > 0 else float('inf')

    print(f'  Batch: {bv} saved, {br} skipped | '
          f'Total: {valid_samples}/{TARGET_SAMPLES} | '
          f'Elapsed: {elapsed/60:.1f}min | ETA: {eta/60:.1f}min')


elapsed = time.time() - start_time

print('\n' + '=' * 60)
print('  GENERATION COMPLETE!')
print('=' * 60)
print(f'  Valid samples   : {valid_samples}')
print(f'  Rejected/skipped: {rejected_samples}')
print(f'  Success rate    : {valid_samples/total_generated*100:.1f}%')
print(f'  Total time      : {elapsed/60:.1f} minutes')
print(f'  Avg per sample  : {elapsed/valid_samples:.2f}s')
print(f'  Files saved to  : {OUTPUT_DIR}')
print('=' * 60)

  AIRI WAKE WORD GENERATION — CHATTERBOX
  Wake word       : 'Hello Airi'
  Target samples  : 10
  Duration range  : 0.8s to 2.0s
  Device          : CUDA
  Sample Rate     : 24000 Hz

--- Epoch 1 | 0/10 done | 10 to go ---


/usr/lib/python3.12/contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)
`sdpa` attention does not support `output_attentions=True`. Please set your attention to `eager` if you want any of these features.
Sampling:   2%|▏         | 24/1000 [00:04<03:06,  5.24it/s]
/usr/lib/python3.12/contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)


  OK  [ 1/10] emo=neutral  spd=0.85 dur=1.13s


Sampling:   2%|▏         | 20/1000 [00:01<01:14, 13.15it/s]


  OK  [ 2/10] emo=happy    spd=1.00 dur=0.80s


...

  OK  [ 9/10] emo=calm     spd=0.90 dur=0.84s


Sampling:   2%|▏         | 20/1000 [00:00<00:29, 32.84it/s]


  SKIP [10/10] 0.73s outside range
  Batch: 9 saved, 1 skipped | Total: 9/10 | Elapsed: 0.9min | ETA: 0.1min

--- Epoch 2 | 9/10 done | 1 to go ---


Sampling:   2%|▎         | 25/1000 [00:01<00:39, 24.66it/s]


  OK  [ 1/1] emo=happy    spd=1.10 dur=0.91s
  Batch: 1 saved, 0 skipped | Total: 10/10 | Elapsed: 0.9min | ETA: 0.0min

  GENERATION COMPLETE!
  Valid samples   : 10
  Rejected/skipped: 1
  Success rate    : 90.9%
  Total time      : 0.9 minutes
  Avg per sample  : 5.55s
  Files saved to  : /content/chatterbox_output


---
## 🔍 Cell 6 — Preview a Random Sample *(Optional)*

Plays a randomly chosen clip from your generated dataset directly in the notebook.
Use this as a quick sanity check before downloading everything.

> 💡 Re-run the cell to hear a different random sample.

---


In [ ]:
from IPython.display import Audio, display
import random as _rnd
from pathlib import Path

wav_files = list(Path(OUTPUT_DIR).glob('*.wav'))
if not wav_files:
    print('No .wav files yet — run Cell 5 first.')
else:
    sample = _rnd.choice(wav_files)
    print(f'Playing: {sample.name}')
    print(f'Total files generated: {len(wav_files)}')
    display(Audio(str(sample)))


Playing: airi_1774782192153_8_calm_spd0.90.wav
Total files generated: 10


---
## 💾 Cell 7 — Download Your Dataset

> ⚠️ **Colab sessions are temporary.** All files are deleted when you disconnect. Download before closing the tab!

| Option | Best for |
|---|---|
| **Option A** — Zip & download | Small to medium datasets (direct browser download) |
| **Option B** — Save to Google Drive | Large datasets (600+ files) or long-term storage |

---


In [ ]:
# ── OPTION A: Download zip directly ──────────────────────────────────────
import zipfile
from google.colab import files
from pathlib import Path

zip_path  = '/content/airi_chatterbox_dataset.zip'
wav_files = list(Path(OUTPUT_DIR).glob('*.wav'))

if not wav_files:
    print('No .wav files found — run Cell 5 first.')
else:
    print(f'Zipping {len(wav_files)} files...')
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for f in wav_files:
            zf.write(f, f.name)
    size_mb = Path(zip_path).stat().st_size / (1024 ** 2)
    print(f'Zip ready: {size_mb:.1f} MB — starting download...')
    files.download(zip_path)


Zipping 10 files...
Zip ready: 0.4 MB — starting download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ── OPTION B: Save to Google Drive (recommended for 600+ files) ──────────
import shutil, os
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
DRIVE_DEST = '/content/drive/MyDrive/airi_chatterbox_dataset'
os.makedirs(DRIVE_DEST, exist_ok=True)

wav_files = list(Path(OUTPUT_DIR).glob('*.wav'))
if not wav_files:
    print('No .wav files found — run Cell 5 first.')
else:
    print(f'Copying {len(wav_files)} files to Google Drive...')
    for f in wav_files:
        shutil.copy(f, DRIVE_DEST)
    print(f'Done! Saved to: My Drive/airi_chatterbox_dataset/')
